# Extraction schemas that pass on the first try

`client.doc_ai.extract()` takes a JSON schema describing the fields you want pulled out
of a document. The rules that schema has to satisfy are written down in exactly one
place, the docstring of the `extract` method inside the installed package, and nothing
in the SDK checks any of them before the request leaves your machine. Every wrong
attempt is a paid, asynchronous round trip: submit, poll, read the rejection, guess
again.

This recipe is the offline check that removes those round trips, plus four ready-made
schemas that already obey the rules.

## Read this before you run anything

Section 6, the only part that calls the live API, **has never been executed**. There is
no Sarvam API key on the machine this was written on, so every SDK call below was
written against signatures read out of the installed `sarvamai` 0.1.30 package, not
against a live response. Nothing here was run, which is why every cell output is empty.
Sections 1 to 5 need no key at all and you can run them right now.

## What the pipeline looks like

    your schema, as a Python dict
      -> schema_lint.lint_schema(json.dumps(schema))     offline, free
      -> schema_lint.check_call_arguments(...)           offline, free
      -> client.doc_ai.extract(...)                      paid, starts an async job
      -> client.doc_ai.get_status(job_id)                poll until terminal
      -> client.doc_ai.get_results(job_id)
      -> schema_lint.find_low_confidence_fields(...)     offline, free

## What this recipe does not ship

No documents. Not one. A bill, a marksheet and an invoice are somebody's private
records, and a generated stand-in would be a made-up artefact dressed as a real one.
The subject here is the schema, not the document: a schema saying where the consumer
number sits on a bill is our own writing, holds nobody's data, and is useful precisely
because you already hold the bill we must never ship. Section 6 reads a file you supply
yourself.

In [ ]:
%pip install -q "sarvamai>=0.1.24" "python-dotenv>=1.0.0"

## 1. The six rules

Quoted from the `extract` docstring in `sarvamai` 0.1.30. These six sentences are the
whole specification the linter enforces, and it enforces nothing else.

1. **Input.** Exactly one of `file` and `upload_ids` must be provided.
2. **Schema source.** Exactly one of `schema` (an inline JSON schema) and `config_id`
   (a saved extraction configuration) is required.
3. **Root shape.** The root must be `type: "object"` with non-empty `properties`, and
   every field needs a `type` and a non-empty `description`.
4. **Types.** `string`, `number`, `integer`, `boolean`, `object`, `array`. Objects need
   `properties`, arrays need `items`. `enum` is optional.
5. **Depth.** Maximum nesting depth 4.
6. **Text booleans.** `classification` and `auto_orient` are booleans sent as text:
   the strings `"true"` and `"false"`.

Two of those bite harder than they read.

`schema` is typed `Optional[str]`, so it wants a **JSON string**, not a dict. Hand it a
dict and the SDK passes it straight into a multipart part, where httpx raises
`AttributeError: 'dict' object has no attribute 'read'`. Nothing in that message
mentions `schema`, and it sends you looking for a file handle. The same thing happens
to a Python `True` in `classification`. Both failures are free, in that no request is
sent, but neither tells you what to change.

That is what `schema_lint` is for: it says which parameter and what to write instead,
before the SDK is ever called.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

RECIPE_DIR = Path.cwd()
if not (RECIPE_DIR / "schema_lint.py").is_file():
    raise RuntimeError(
        "Run this notebook from inside examples/doc-extraction-schemas, "
        "so that schema_lint.py sits next to it."
    )

sys.path.insert(0, str(RECIPE_DIR))
import schema_lint

print("maximum nesting depth:", schema_lint.MAX_DEPTH)
print("codes the linter can emit:", len(schema_lint.FINDING_CODES))

## 2. A schema with five things wrong

This is roughly what a first attempt looks like. Every one of these would come back as
a rejection from the API, one paid round trip at a time.

In [ ]:
BROKEN_SCHEMA = {
    "type": "object",
    "description": "an electricity bill",
    "properties": {
        "consumer_name": {"description": "the name printed at the top"},
        "units_consumed": {"type": "float", "description": "units billed this period"},
        "billing_period": {"type": "string", "description": "   "},
        "supply_address": {"type": "object", "description": "where the meter sits"},
        "tariff_slabs": {"type": "array", "description": "the slab rows"},
    },
}

for finding in schema_lint.lint_schema(json.dumps(BROKEN_SCHEMA)):
    print(f"[{finding.severity}] {finding.code}  {finding.path}")
    print(f"    {finding.message}")
    print(f"    fix: {finding.suggestion}")

## 3. The same schema, fixed

`float` becomes `number`, the blank description gets written, the object gets
`properties` and the array gets `items`.

### How depth is counted here

The docstring says "maximum nesting depth 4" and never says how depth is counted, so
this recipe picks a convention and states it in the open: **the root object is depth 1,
stepping into `properties.<name>` adds 1, stepping into `items` adds 1, and depth 5 is
an error.**

    {                                     root object          depth 1
      "properties": {
        "consumer_name": {...}                                 depth 2
        "tariff_slabs": {"type": "array",                      depth 2
          "items": {"type": "object",                          depth 3
            "properties": {
              "units": {"type": "number"}                      depth 4   allowed
            }}}}}

**This convention has not been confirmed against the live API.** If the server counts
the root as depth 0, this linter is one level stricter than it needs to be and will
occasionally tell you to flatten something the server would have taken. That costs you
an edit. Being too lenient in the other direction costs a paid round trip, which is the
thing this recipe exists to prevent, so where the reading is ambiguous the stricter one
wins. `MAX_DEPTH` is a named constant in `schema_lint.py`: if you learn the real
convention from the API, change that one line.

In [ ]:
FIXED_SCHEMA = {
    "type": "object",
    "description": "an electricity bill",
    "properties": {
        "consumer_name": {
            "type": "string",
            "description": "the name printed at the top, above the supply address",
        },
        "units_consumed": {
            "type": "number",
            "description": "total units billed this period, in kWh",
        },
        "billing_period": {
            "type": "string",
            "description": "the from and to dates this bill covers",
        },
        "supply_address": {
            "type": "object",
            "description": "where the meter sits",
            "properties": {
                "line1": {"type": "string", "description": "door number and street"},
                "pin_code": {
                    "type": "string",
                    "description": "six digit postal code, kept as text so zeros survive",
                },
            },
        },
        "tariff_slabs": {
            "type": "array",
            "description": "one entry per row of the energy charge table",
            "items": {
                "type": "object",
                "description": "a single slab row",
                "properties": {
                    "units": {
                        "type": "number",
                        "description": "units billed inside this slab",
                    },
                    "rate_per_unit": {
                        "type": "number",
                        "description": "rate charged per unit in this slab, in rupees",
                    },
                },
            },
        },
    },
}

print("findings:", schema_lint.lint_schema(json.dumps(FIXED_SCHEMA)))

Now push it one level past the limit. `units` becomes an object, so its child lands at
depth 5 and the linter names the full counted path.

In [ ]:
TOO_DEEP = json.loads(json.dumps(FIXED_SCHEMA))
TOO_DEEP["properties"]["tariff_slabs"]["items"]["properties"]["units"] = {
    "type": "object",
    "description": "units billed inside this slab",
    "properties": {
        "value": {"type": "number", "description": "the number itself"},
    },
}

for finding in schema_lint.lint_schema(json.dumps(TOO_DEEP)):
    print(f"{finding.code}: {finding.message}")

## 4. Four schemas you can start from

`schemas/` holds four schemas for documents an Indian reader is likely to have on hand:
an electricity bill, a school marksheet, a pharmacy invoice and an LPG refill receipt.
Each one describes where the fields sit. None of them holds anybody's data, and none of
them is filled in.

They are checked by the test suite on every run, so any one of them that stops linting
cleanly breaks the build.

In [ ]:
for path in sorted((RECIPE_DIR / "schemas").glob("*.json")):
    schema = json.loads(path.read_text(encoding="utf-8"))
    findings = schema_lint.lint_schema(json.dumps(schema))
    fields = len(schema["properties"])
    print(f"{path.name:<26} {fields:>2} top-level fields   {len(findings)} finding(s)")

The same check runs from a terminal, over as many files as you like:

    python schema_lint.py schemas/electricity_bill.json
    python schema_lint.py schemas/*.json --json

It exits 0 only when every file lints clean, so it drops straight into a pre-commit
hook or a CI step.

## 5. The call itself

`lint_schema` looks at the schema. `check_call_arguments` looks at the whole call, and
catches the two traps that are not really schema problems at all: a dict where a JSON
string belongs, and a Python `True` where the text `"true"` belongs.

In [ ]:
wrong = schema_lint.check_call_arguments(
    file=["bill.pdf"],
    upload_ids="upload_1234",
    schema=FIXED_SCHEMA,
    language="english",
    output_format="pdf",
    classification=True,
)
for finding in wrong:
    print(f"[{finding.severity}] {finding.code}  {finding.path}: {finding.message}")

print()

right = schema_lint.check_call_arguments(
    file=["bill.pdf"],
    schema=json.dumps(FIXED_SCHEMA),
    language="en-IN",
    output_format="json",
    classification="true",
    auto_orient="true",
)
print("findings on the corrected call:", right)

Note what changed between the two calls, because these are the edits that matter:

- `upload_ids` dropped. Exactly one input source, never both.
- `schema=json.dumps(FIXED_SCHEMA)`, a string. Never the dict.
- `language="en-IN"`, not `"english"`. Region uppercase, language lowercase, a hyphen
  between them.
- `output_format="json"`. The installed SDK types exactly three: `json`, `csv`, `xlsx`.
- `classification="true"`, the string. Never a Python `True`.

The language check is shape-only on purpose. This repo has no verified list of the
languages document extraction accepts, so a well-formed tag outside India comes back as
a warning rather than an error. Inventing an allowlist would be guessing, and there is
already a recorded case in this repo of a language code that the rules file permits and
the API rejects.

## The confidence gate

An extract result carries `result` (the values) and `annotations` (per-field
confidence). `find_low_confidence_fields` walks the annotations and hands back
everything below your threshold, worst first, with dotted paths and array indices, so
you know exactly which fields to check by hand.

**The fixture below is authored by us**, in the shape the `sarvamai` 0.1.30 docstring
describes. It was never captured from a live response. The `annotations` field is typed
`Dict[str, Any]` and no model in the SDK pins what is inside it, so the shape is prose,
not a guarantee. Check it against your first real response.

One deliberate piece of rudeness: if the payload holds no `confidence` anywhere, this
raises instead of returning an empty list. The most likely way to get that wrong is to
hand it `result` instead of `annotations`, and a helper that answered "nothing is low
confidence" to a shape it did not understand would defeat the entire point.

In [ ]:
ANNOTATIONS_EXAMPLE = {
    "consumer_name": {"confidence": 0.97, "sources": [{"page": 1}]},
    "billing_period": {"confidence": 0.55, "sources": [{"page": 1}]},
    "supply_address": {
        "line1": {"confidence": 0.91, "sources": [{"page": 1}]},
        "pin_code": {"confidence": 0.80, "sources": [{"page": 1}]},
    },
    "tariff_slabs": [
        {"units": {"confidence": 0.88, "sources": [{"page": 2}]}},
        {"units": {"confidence": 0.42, "sources": [{"page": 2}]}},
    ],
}

for path, score in schema_lint.find_low_confidence_fields(ANNOTATIONS_EXAMPLE, 0.80):
    print(f"check by hand: {path} at {score}")

try:
    schema_lint.find_low_confidence_fields({"consumer_name": "a name"}, 0.80)
except ValueError as error:
    print("\nhanded the wrong half of the response:", error)

## 6. The live call

**This section has never been run.** There is no Sarvam API key on the machine this
notebook was written on, so nothing below was executed and nothing below has a saved
output. The calls follow the signatures in `sarvamai` 0.1.30 as installed, and that is
the strongest claim this recipe makes about them. Treat the first run as the real test.

Two things you have to supply:

1. A key. Copy `.env.example` to `.env` and fill it in.
2. A document. Put your own scan at `sample_data/your-bill.pdf`. This recipe ships no
   documents and will not generate one.

Note what is missing from the call: there is no `model` argument. It is optional, this
repo has no verified value for it, and writing a plausible-looking one would be making
something up. Leaving it out is both correct and safe.

The job is asynchronous, so the pattern is submit, poll `get_status` until the status
is one of `completed`, `partially_completed`, `failed` or `rejected`, then read
`get_results`.

In [ ]:
import os
import time

from dotenv import load_dotenv
from sarvamai import SarvamAI

load_dotenv()

if not os.environ.get("SARVAM_API_KEY"):
    raise RuntimeError(
        "No API key found. Copy .env.example to .env and put your key in it."
    )

DOCUMENT_PATH = Path("sample_data/your-bill.pdf")   # you supply this
if not DOCUMENT_PATH.is_file():
    raise RuntimeError(
        f"No document at {DOCUMENT_PATH}. Put your own scan there. "
        "This recipe ships no documents and will not invent one."
    )

# The key is passed explicitly on purpose. The client's own default is
# os.getenv(...) evaluated once at import time, so load_dotenv() afterwards
# is too late and the default is already None.
client = SarvamAI(api_subscription_key=os.environ["SARVAM_API_KEY"])

call_arguments = {
    "file": [(DOCUMENT_PATH.name, DOCUMENT_PATH.read_bytes(), "application/pdf")],
    "schema": json.dumps(FIXED_SCHEMA),
    "language": "en-IN",
    "output_format": "json",
    "classification": "false",
    "auto_orient": "true",
}

problems = schema_lint.check_call_arguments(**call_arguments)
if problems:
    for finding in problems:
        print(f"[{finding.severity}] {finding.code}  {finding.path}: {finding.message}")
    raise RuntimeError("Fix the findings above before spending a request.")

job = client.doc_ai.extract(**call_arguments)
print("job id:", job.job_id)

TERMINAL = {"completed", "partially_completed", "failed", "rejected"}
while True:
    job_status = client.doc_ai.get_status(job.job_id)
    print("status:", job_status.status)
    if job_status.status in TERMINAL:
        break
    time.sleep(5)

if job_status.status in {"failed", "rejected"}:
    raise RuntimeError(
        f"The job ended as {job_status.status}, so there is nothing to read."
    )

results = client.doc_ai.get_results(job.job_id)
print(json.dumps(results.result, indent=2, ensure_ascii=False))

for path, score in schema_lint.find_low_confidence_fields(results.annotations, 0.80):
    print(f"check by hand: {path} at {score}")

## What to do next

- Copy one of the four schemas out of `schemas/` and edit it for your own document.
  Run `python schema_lint.py your_schema.json` until it comes back clean.
- Keep the confidence gate in the loop. A field the model was unsure about is the one
  worth a human glance, and the threshold is yours to set.
- If the API rejects a schema this linter passed, or accepts one it rejected, that is
  worth reporting. The depth convention in particular is our reading of an ambiguous
  sentence, not a confirmed fact.